# Fitting: log-poly and PLS κ\_m surrogates

This notebook produces (or loads, if cached) two surrogate model artifacts:

- **Log-poly model**: per-wavenumber coefficients saved under `fit_cache/<species>_fit_data.npz`
- **PLS model**: per-column predicted κ\_m arrays saved as `kappas_pred_<species>_PLS.npz`

In [1]:
import os
import numpy as np
import xarray as xr
import pyarts

from util import (
    MOLAR_MASS,
    PLS_WVN_GROUPS,
    make_sampling_grids_from_envelope,
    report_sampling_vs_envelope,
    generate_kappam_grid_for_species,
    fit_logpoly_per_wavenumber,
    train_pls_kappa_arrays,
    compute_global_cost_over_all_columns_surrogate,
    compute_global_cost_over_all_columns_surrogate_coeffs,
)
# Ensure spectroscopy tables are available
pyarts.cat.download.retrieve()

## Inputs

In [2]:
profiles_full = xr.open_dataset("ckdmip_evaluation1_concentrations_present.nc")
DDQ_lw = xr.open_dataset("DDQ_LW_present.h5", engine="netcdf4")
DDQ_lw.compute()

nu_cm1 = DDQ_lw.S.values.astype(float)
n_cols = int(profiles_full.sizes["column"])
n_levels = int(profiles_full.sizes["half_level"])
print(f"DDQ ν points: {nu_cm1.size} | columns: {n_cols} | levels: {n_levels}")

DDQ ν points: 64 | columns: 50 | levels: 55


## Training grids (T×p coverage)

In [3]:
T_set, p_hl_grid = make_sampling_grids_from_envelope(
    profiles_full, nT=9, nP=55, T_pad=0.05, p_pad=0.05, p_top_floor=1e-2
)
report_sampling_vs_envelope(profiles_full, T_set, p_hl_grid, p_top_floor=1e-2)

CKDMIP envelope: p∈[1.00e-02,1.03e+05] Pa, T∈[169.1,326.2] K
Sampling grids : p∈[1.00e-02,1.08e+05] Pa (surf→TOA), T∈[160.6,342.5] K
✅ Sampling covers CKDMIP and extends a little outside.


## Cache paths

In [4]:
FIT_CACHE_DIR = "fit_cache"
os.makedirs(FIT_CACHE_DIR, exist_ok=True)

## Step 1 — Build or load κ\_m(T,p) training grids (ARTS)

We cache the raw κ\_m training grid so both the log-poly fit and PLS training can reuse it.

In [5]:
def load_or_build_kappa_grid(species, vmr_value=1e-4):
    cache_path = os.path.join(FIT_CACHE_DIR, f"{species.lower()}_kappa_grid.npz")
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        kappa = data["kappa_m_wvn_pT"]
        P_used = data["P_used"]
        T_used = data["T_used"]
        nu_saved = data["nu_cm1"]
        if nu_saved.shape != nu_cm1.shape or np.max(np.abs(nu_saved - nu_cm1)) > 1e-9:
            raise RuntimeError(f"{cache_path} ν grid does not match current DDQ grid.")
        return kappa, P_used, T_used

    print(f"[{species}] ARTS sampling κ_m(T,p) grid (this can take a while)...")
    kappa, P_used, T_used = generate_kappam_grid_for_species(
        species=species,
        wvn_cm1=nu_cm1,
        p_hl_grid_Pa=p_hl_grid,
        T_set_K=T_set,
        vmr_value=vmr_value,
    )
    np.savez_compressed(cache_path, kappa_m_wvn_pT=kappa, P_used=P_used, T_used=T_used, nu_cm1=nu_cm1)
    print(f"[{species}] ✓ Saved κ_m grid to {cache_path}")
    return kappa, P_used, T_used

## Step 2 — Fit or load log-poly coefficients

In [6]:
def fit_or_load_logpoly(species, kappa_grid, P_used, T_used, ridge=1e-6):
    cache_path = os.path.join(FIT_CACHE_DIR, f"{species.lower()}_fit_data.npz")
    if os.path.exists(cache_path):
        print(f"[{species}] Loading log-poly fit from {cache_path}")
        return np.load(cache_path)["coeffs"]

    print(f"[{species}] Fitting log-poly coefficients...")
    coeffs = fit_logpoly_per_wavenumber(kappa_grid, P_used, T_used, ridge=ridge)
    np.savez_compressed(cache_path, coeffs=coeffs, P_used=P_used, T_used=T_used, nu=nu_cm1)
    print(f"[{species}] ✓ Saved log-poly fit to {cache_path}")
    return coeffs

In [7]:
logpoly_coeffs = {}
for sp in ("CO2", "O3"):
    kappa_grid, P_used, T_used = load_or_build_kappa_grid(sp)
    logpoly_coeffs[sp] = fit_or_load_logpoly(sp, kappa_grid, P_used, T_used)

[CO2] Loading log-poly fit from fit_cache/co2_fit_data.npz
[O3] Loading log-poly fit from fit_cache/o3_fit_data.npz


In [8]:
cost_logpoly_vs_ddq = compute_global_cost_over_all_columns_surrogate_coeffs(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=logpoly_coeffs,
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),
    ddq_cache_dir="lbl_ddq",
)
print(f"[log-poly] global cost vs DDQ: {cost_logpoly_vs_ddq:.6f}")


/srv/conda/envs/ddq_fluxsim_tutorial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Surrogate vs DDQ (coeffs): 100%|███████████████| 50/50 [00:00<00:00, 136.06it/s]

[log-poly] global cost vs DDQ: 31.838688


## Step 3 — Train or load PLS κ\_m predictions for CKDMIP columns (Bad Performance)

In [9]:
test_T = profiles_full.temperature_hl.values.astype(float)  # (n_col, n_level)
test_P = profiles_full.pressure_hl.values.astype(float)     # (n_col, n_level)

In [10]:
def fit_or_load_pls_arrays(species, kappa_grid, P_used, T_used):
    out_path = os.path.join(FIT_CACHE_DIR, f"kappas_pred_{species}_PLS.npz")
    if os.path.exists(out_path):
        print(f"[{species}] Loading PLS κ arrays from {out_path}")
        with np.load(out_path) as data:
            return data["data1"]

    spec = PLS_WVN_GROUPS[species]
    print(f"[{species}] Training PLS per wavenumber and predicting κ arrays...")
    kappa_pred = train_pls_kappa_arrays(
        kappa_train_wvn_pT=kappa_grid,
        P_train_Pa=P_used,
        T_train_K=T_used,
        test_pressure_Pa=test_P,
        test_temperature_K=test_T,
        wvn_zeros=spec["zeros"],
        wvn_by_power=spec["by_power"],
        n_components=spec.get("n_components", 2),
        log10_floor=1e-30,
    )
    np.savez_compressed(out_path, data1=kappa_pred)
    print(f"[{species}] ✓ Saved PLS κ arrays to {out_path}")
    return kappa_pred

In [11]:
pls_arrays = {}
for sp in ("CO2", "O3"):
    kappa_grid, P_used, T_used = load_or_build_kappa_grid(sp)
    pls_arrays[sp] = fit_or_load_pls_arrays(sp, kappa_grid, P_used, T_used)
    print(f"[{sp}] κ_pred shape: {pls_arrays[sp].shape}")

[CO2] Loading PLS κ arrays from fit_cache/kappas_pred_CO2_PLS.npz
[CO2] κ_pred shape: (64, 50, 55)
[O3] Loading PLS κ arrays from fit_cache/kappas_pred_O3_PLS.npz
[O3] κ_pred shape: (64, 50, 55)


In [12]:
cost_pls_vs_ddq = compute_global_cost_over_all_columns_surrogate(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_arrays_dict=pls_arrays,
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),
    ddq_cache_dir="lbl_ddq",
)
print(f"[PLS arrays] global cost vs DDQ: {cost_pls_vs_ddq:.6f}")

Surrogate vs DDQ (arrays): 100%|███████████████| 50/50 [00:00<00:00, 138.45it/s]

[PLS arrays] global cost vs DDQ: 10780.849651
